# 01 — Matching & Loading

**Network Medicine Workshop · Kidney Disease · Part 1 of 3 (+ optional Part 4)**

This notebook handles the unglamorous-but-essential part of network medicine: getting *your* messy omics lists onto standardized IDs that match the node IDs used in the networks, and loading + sanity-checking those networks.

You're bringing **three independent datasets**, each from a different omics layer:
- **Transcripts** — DEGs from spatial transcriptomics
- **Proteins** — DE proteins from proteomics
- **Metabolites** — DE metabolites from metabolomics

Each gets matched to the ID system used by its corresponding network: transcripts and proteins both resolve to **NCBI Gene IDs** (transcripts → transcriptome network, proteins → PPI network), metabolites resolve to **KEGG Compound IDs** (→ metabolite network).

**What you'll do here (~15–20 min):**
1. Load all three DE lists
2. Batch-map gene/protein symbols → NCBI Gene IDs (with a fuzzy fallback for typos/old symbols)
3. Batch-map metabolite names → KEGG Compound IDs (local KEGG dictionary + vectorized fuzzy matching)
4. Load each network from your Drive `networks/` folder and print basic stats
5. Save everything to `processed/` so later notebooks can pick it up

**You will need, in your Drive workshop folder:**
- `data/DEGs.csv` — needs a `gene_symbol` column (transcripts)
- `data/DE_proteins.csv` — needs a `protein_symbol` column (the gene symbol encoding each protein; extra columns like fold-change kept)
- `data/DE_metabolites.csv` — needs a `metabolite_name` column
- `networks/ppi_network.csv` — edges `source,target[,weight]`, **NCBI Gene IDs**
- `networks/transcriptome_network.csv` — same format, **NCBI Gene IDs**
- `networks/metabolite_network.csv` — same format, **KEGG Compound IDs**

If your column names differ, just rename them below in the config cell — nothing else needs to change.


## Setup

In [5]:
# Install what we need. mygene = batch gene ID mapping, rapidfuzz = fast fuzzy string matching.
!pip install -q mygene rapidfuzz networkx requests tqdm pandas



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [3]:
# ---- CONFIG: adjust these paths/column names to match your setup ----
import os

BASE_DIR = "./"   # <- change if your folder is named differently

DATA_DIR = os.path.join(BASE_DIR, "data")
NET_DIR = os.path.join(BASE_DIR, "networks")
PROC_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(PROC_DIR, exist_ok=True)

DEG_FILE = os.path.join(DATA_DIR, "DEGs.csv")               # transcripts
PROTEIN_FILE = os.path.join(DATA_DIR, "DE_proteins.csv")    # proteins
METAB_FILE = os.path.join(DATA_DIR, "DE_metabolites.csv")   # metabolites

GENE_SYMBOL_COL = "gene_symbol"
PROTEIN_SYMBOL_COL = "protein_symbol"
METAB_NAME_COL = "metabolite_name"

# network layer -> (filename, which dataset maps onto it)
NETWORK_FILES = {
    "transcriptome": "transcriptome_network.csv",   # matched against DEGs (transcripts), NCBI Gene IDs
    "ppi":           "ppi_network.csv",             # matched against DE proteins, NCBI Gene IDs
    "metabolite":    "metabolite_network.csv",      # matched against DE metabolites, KEGG Compound IDs
}

print("Base dir:", BASE_DIR)


Base dir: ./


## Step 1 — Load your three lists

Nothing fancy here, just load and take a quick look at each. This is also your chance to catch obvious problems (wrong column names, encoding issues, duplicate entries) before they propagate downstream.


In [14]:
import pandas as pd

degs = pd.read_csv(DEG_FILE)
proteins = pd.read_csv(PROTEIN_FILE)
metabs = pd.read_csv(METAB_FILE)

# Basic hygiene: strip whitespace, drop exact duplicates
degs[GENE_SYMBOL_COL] = degs[GENE_SYMBOL_COL].astype(str).str.strip()
proteins[PROTEIN_SYMBOL_COL] = proteins[PROTEIN_SYMBOL_COL].astype(str).str.strip()
metabs[METAB_NAME_COL] = metabs[METAB_NAME_COL].astype(str).str.strip()

degs = degs.drop_duplicates(subset=GENE_SYMBOL_COL).reset_index(drop=True)
proteins = proteins.drop_duplicates(subset=PROTEIN_SYMBOL_COL).reset_index(drop=True)
metabs = metabs.drop_duplicates(subset=METAB_NAME_COL).reset_index(drop=True)

print(f"Transcripts (DEGs):    {len(degs)} unique gene symbols")
print(f"Proteins (proteomics): {len(proteins)} unique protein/gene symbols")
print(f"Metabolites:           {len(metabs)} unique metabolite names")
degs.head()


ParserError: Error tokenizing data. C error: Expected 1 fields in line 36, saw 2


In [ ]:
proteins.head()


In [ ]:
metabs.head()


## Step 2 — Gene/protein symbol → NCBI Gene ID

Transcripts and proteins are both matched the same way, since both ultimately resolve to a gene identifier — we just do it as **one batch call per dataset** rather than looping symbol-by-symbol. `mygene.info`'s `querymany` accepts up to ~1000 identifiers per request and resolves current symbols *and* aliases/old names in a single pass. This is the single biggest speed difference vs. the naive approach.

Anything still unmatched after that gets a fuzzy-matching pass against the pool of symbols the batch call *did* resolve, so a typo like `"HAVCR-1"` still finds `HAVCR1`.

We wrap this in one reusable function and call it twice — once for transcripts, once for proteins — so both datasets get identical treatment.


In [ ]:
import mygene
from rapidfuzz import process, fuzz

mg = mygene.MyGeneInfo()

def match_symbols_to_ncbi(symbols, label):
    """Batch-map a list of gene/protein symbols to NCBI Gene IDs, with fuzzy fallback."""
    result = mg.querymany(
        symbols, scopes="symbol,alias", fields="entrezgene,symbol",
        species="human", returnall=True,
    )

    matched_rows = []
    seen = set()
    for hit in result["out"]:
        if "entrezgene" in hit and hit["query"] not in seen:
            matched_rows.append({
                "query": hit["query"],
                "ncbi_gene_id": str(hit["entrezgene"]),
                "matched_symbol": hit.get("symbol", ""),
                "match_type": "exact/alias",
                "match_score": 100,
            })
            seen.add(hit["query"])

    matches = pd.DataFrame(matched_rows).drop_duplicates(subset="query")
    missing = sorted(set(symbols) - set(matches["query"]))
    print(f"[{label}] matched directly: {len(matches)} / {len(symbols)}  |  unmatched: {len(missing)}")

    if missing:
        reference_symbols = matches["matched_symbol"].tolist()
        fuzzy_rows, still_missing = [], []
        for q in missing:
            best = process.extractOne(q, reference_symbols, scorer=fuzz.WRatio)
            if best and best[1] >= 85:
                row = matches.loc[matches["matched_symbol"] == best[0]].iloc[0]
                fuzzy_rows.append({
                    "query": q, "ncbi_gene_id": row["ncbi_gene_id"],
                    "matched_symbol": best[0], "match_type": "fuzzy", "match_score": best[1],
                })
            else:
                still_missing.append(q)
        matches = pd.concat([matches, pd.DataFrame(fuzzy_rows)], ignore_index=True)
        print(f"[{label}] recovered via fuzzy matching (score>=85): {len(fuzzy_rows)}  |  "
              f"genuinely unmatched: {len(still_missing)}")
        if still_missing:
            print(f"[{label}] unmatched examples:", still_missing[:15])

    return matches

gene_matches = match_symbols_to_ncbi(degs[GENE_SYMBOL_COL].tolist(), "transcripts")
protein_matches = match_symbols_to_ncbi(proteins[PROTEIN_SYMBOL_COL].tolist(), "proteins")


In [ ]:
gene_matches.sort_values("match_score").head(10)   # eyeball the shakiest transcript matches


In [ ]:
protein_matches.sort_values("match_score").head(10)   # eyeball the shakiest protein matches


In [ ]:
degs_matched = degs.merge(
    gene_matches[["query", "ncbi_gene_id", "matched_symbol", "match_type", "match_score"]],
    left_on=GENE_SYMBOL_COL, right_on="query", how="left"
).drop(columns="query")

proteins_matched = proteins.merge(
    protein_matches[["query", "ncbi_gene_id", "matched_symbol", "match_type", "match_score"]],
    left_on=PROTEIN_SYMBOL_COL, right_on="query", how="left"
).drop(columns="query")

print(f"Transcripts: {degs_matched['ncbi_gene_id'].notna().sum()} / {len(degs_matched)} matched")
print(f"Proteins:    {proteins_matched['ncbi_gene_id'].notna().sum()} / {len(proteins_matched)} matched")


## Step 3 — Metabolite name → KEGG Compound ID

Same philosophy, different resource: don't hit an API per metabolite. Instead we download the **full KEGG compound list once** (~19k entries, one request) and build a local name/synonym lookup. Exact matches resolve instantly; everything else goes through a single vectorized fuzzy-matching pass (`rapidfuzz.process.cdist`), which compares all your unmatched names against all KEGG names in one shot instead of nested loops.

Matches are scored so you can see, at a glance, which ones need a human to double check — this is the realistic way to handle "someone else's" metabolite naming conventions (brand names, plurals, stereochemistry prefixes, etc.).


In [ ]:
import requests

resp = requests.get("https://rest.kegg.jp/list/compound", timeout=60)
resp.raise_for_status()

kegg_id_to_names = {}
for line in resp.text.strip().split("\n"):
    cid, names = line.split("\t")
    cid = cid.replace("cpd:", "").strip()
    kegg_id_to_names[cid] = [n.strip() for n in names.split(";")]

print(f"Loaded {len(kegg_id_to_names)} KEGG compounds")

# Flat lookup: lowercase name -> KEGG ID (first name wins if there's a collision)
name_to_kegg = {}
for cid, names in kegg_id_to_names.items():
    for n in names:
        name_to_kegg.setdefault(n.lower(), cid)

all_kegg_names = list(name_to_kegg.keys())


In [ ]:
metab_names = metabs[METAB_NAME_COL].tolist()

exact_rows, unmatched_metabs = [], []
for name in metab_names:
    key = name.lower()
    if key in name_to_kegg:
        exact_rows.append({
            "query": name, "kegg_id": name_to_kegg[key],
            "matched_name": name, "match_type": "exact", "match_score": 100
        })
    else:
        unmatched_metabs.append(name)

print(f"Exact matches: {len(exact_rows)} / {len(metab_names)}")
print(f"Sent to fuzzy matching: {len(unmatched_metabs)}")


In [ ]:
fuzzy_rows = []
if unmatched_metabs:
    import numpy as np
    # cdist = compute a full similarity matrix in one vectorized call (fast, C-backed)
    scores = process.cdist(unmatched_metabs, all_kegg_names, scorer=fuzz.WRatio, workers=-1)
    best_idx = scores.argmax(axis=1)
    best_scores = scores.max(axis=1)

    for name, idx, score in zip(unmatched_metabs, best_idx, best_scores):
        matched_name = all_kegg_names[idx]
        fuzzy_rows.append({
            "query": name, "kegg_id": name_to_kegg[matched_name],
            "matched_name": matched_name, "match_type": "fuzzy", "match_score": round(float(score), 1),
        })

metab_matches = pd.DataFrame(exact_rows + fuzzy_rows)

# Human-in-the-loop bands: >=90 auto-accept, 70-89 review, <70 likely wrong
metab_matches["review_flag"] = pd.cut(
    metab_matches["match_score"], bins=[0, 69.999, 89.999, 100],
    labels=["reject/manual", "review", "auto-accept"]
)
print(metab_matches["review_flag"].value_counts())
metab_matches.sort_values("match_score").head(15)


In [ ]:
# --- Manual review step ---
# Anything flagged "review" or "reject/manual" below is worth a human glance.
# Edit the dict below to fix specific entries, then re-run this cell.
manual_overrides = {
    # "your messy name here": "C00031",
}

for name, cid in manual_overrides.items():
    mask = metab_matches["query"] == name
    metab_matches.loc[mask, ["kegg_id", "match_type", "match_score", "review_flag"]] = [cid, "manual", 100, "auto-accept"]

metab_matches[metab_matches["review_flag"] != "auto-accept"]


In [ ]:
metabs_matched = metabs.merge(
    metab_matches[["query", "kegg_id", "match_type", "match_score", "review_flag"]],
    left_on=METAB_NAME_COL, right_on="query", how="left"
).drop(columns="query")

print(f"{metabs_matched['kegg_id'].notna().sum()} / {len(metabs_matched)} metabolites matched to a KEGG ID")
metabs_matched.head()


## Step 4 — Load & screen the networks

Just loading edge lists and reporting the numbers every network-medicine analysis should start with: how many nodes, how many edges, how dense, how many connected components, and how big the largest one is. At up to ~20k nodes these are all cheap (linear-time) operations in `networkx` — the thing to avoid later is anything that scales quadratically (e.g. all-pairs shortest paths), which we'll sidestep in later notebooks.


In [ ]:
import networkx as nx
import time

graphs = {}
for layer, fname in NETWORK_FILES.items():
    path = os.path.join(NET_DIR, fname)
    t0 = time.time()
    edges = pd.read_csv(path)
    edges.columns = [c.strip().lower() for c in edges.columns]
    assert {"source", "target"}.issubset(edges.columns), f"{fname} needs 'source' and 'target' columns"

    if "weight" in edges.columns:
        G = nx.from_pandas_edgelist(edges, "source", "target", edge_attr="weight")
    else:
        G = nx.from_pandas_edgelist(edges, "source", "target")

    # Node IDs from CSV often come in as int64/float64/str inconsistently - normalize to str
    G = nx.relabel_nodes(G, {n: str(n) for n in G.nodes()})

    graphs[layer] = G
    n_cc = nx.number_connected_components(G)
    largest_cc = len(max(nx.connected_components(G), key=len))
    print(f"[{layer}] {fname}")
    print(f"   nodes={G.number_of_nodes():,}  edges={G.number_of_edges():,}  "
          f"density={nx.density(G):.5f}  components={n_cc}  largest_cc={largest_cc:,} "
          f"({largest_cc/G.number_of_nodes():.1%} of nodes)")
    print(f"   loaded in {time.time()-t0:.1f}s\n")


**What to look for:** a healthy PPI/co-expression network usually has one dominant connected component covering the large majority of nodes — if your largest component is small relative to the total, either the network is unusually fragmented or something's off in the edge list (e.g. duplicate ID systems mixed together). Worth flagging to the group live if it looks off.


## Step 5 — Save everything for the next notebooks

In [ ]:
import pickle

for layer, G in graphs.items():
    with open(os.path.join(PROC_DIR, f"graph_{layer}.pkl"), "wb") as f:
        pickle.dump(G, f)

degs_matched.to_csv(os.path.join(PROC_DIR, "degs_matched.csv"), index=False)
proteins_matched.to_csv(os.path.join(PROC_DIR, "proteins_matched.csv"), index=False)
metabs_matched.to_csv(os.path.join(PROC_DIR, "metabolites_matched.csv"), index=False)

print("Saved to", PROC_DIR)
print(os.listdir(PROC_DIR))


---
**Next:** open `02_overlay_enrichment.ipynb` — it picks up exactly where this notebook left off. There's also an **optional Notebook 4** (`04_disease_modules_optional.ipynb`) for a deeper dive into module significance and comparing your data against other diseases, if you have time for it.
